<a href="https://colab.research.google.com/github/kjahan/llm_limitations/blob/main/notebooks/json_dsl_converrters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Haullacination examples

In [1]:
!pip3 install --upgrade openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 1.1 MB/s eta 0:00:00


In [2]:
import openai
import getpass

from io import StringIO
from contextlib import redirect_stdout

## API Key

In [3]:
try:
    openai.api_key = getpass.getpass()
except Exception as error:
    print('ERROR', error)

··········


## Call GPT

In [4]:
MODEL = "gpt-3.5-turbo"
system_promp = "You are a professional assistant who answers questions based on facts. If the question is tricky, non-sensical or incomplete respond with Unknown. Only answer a question if you are very confident."

## `Native JSON Output From GPT-4`

https://yonom.substack.com/p/native-json-output-from-gpt-4

https://news.ycombinator.com/item?id=36330972

https://github.com/1rgs/jsonformer

In [5]:
schema = {
  "type": "object",
  "properties": {
    "ingredients": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "name": { "type": "string" },
          "unit": {
            "type": "string",
            "enum": ["grams", "ml", "cups", "pieces", "teaspoons"]
          },
          "amount": { "type": "number" }
        },
        "required": ["name", "unit", "amount"]
      }
    },
    "instructions": {
      "type": "array",
      "description": "Steps to prepare the recipe (no numbering)",
      "items": { "type": "string" }
    },
    "time_to_cook": {
      "type": "number",
      "description": "Total time to prepare the recipe in minutes"
    }
  },
  "required": ["ingredients", "instructions", "time_to_cook"]
}

## API call
Now, let’s call the OpenAI API and pass the JSON schema defined above.


In [8]:
completion = openai.ChatCompletion.create(
  model="gpt-4-0613",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Provide a recipe for chelo kabab koobideh"}
  ],
  functions=[{"name": "set_recipe", "parameters": schema}],
  function_call={"name": "set_recipe"},
  temperature=0,
)

In [9]:
print(completion.choices[0].message.function_call.arguments)

{
  "ingredients": [
    {
      "name": "Ground lamb",
      "unit": "grams",
      "amount": 500
    },
    {
      "name": "Ground beef",
      "unit": "grams",
      "amount": 500
    },
    {
      "name": "Onions",
      "unit": "pieces",
      "amount": 2
    },
    {
      "name": "Garlic cloves",
      "unit": "pieces",
      "amount": 2
    },
    {
      "name": "Salt",
      "unit": "teaspoons",
      "amount": 2
    },
    {
      "name": "Black pepper",
      "unit": "teaspoons",
      "amount": 1
    },
    {
      "name": "Turmeric",
      "unit": "teaspoons",
      "amount": 1
    },
    {
      "name": "Saffron",
      "unit": "teaspoons",
      "amount": 0.5
    },
    {
      "name": "Eggs",
      "unit": "pieces",
      "amount": 2
    },
    {
      "name": "Bread crumbs",
      "unit": "cups",
      "amount": 0.5
    }
  ],
  "instructions": [
    "Grate the onions and squeeze out the excess juice.",
    "In a large bowl, combine the ground lamb, ground beef, gra

In [10]:
completion = openai.ChatCompletion.create(
  model="gpt-4-0613",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Provide a recipe for Galbi-jjim"}
  ],
  functions=[{"name": "set_recipe", "parameters": schema}],
  function_call={"name": "set_recipe"},
  temperature=0,
)

In [11]:
print(completion.choices[0].message.function_call.arguments)

{
  "ingredients": [
    {
      "name": "beef short ribs",
      "unit": "grams",
      "amount": 1000
    },
    {
      "name": "soy sauce",
      "unit": "ml",
      "amount": 120
    },
    {
      "name": "sugar",
      "unit": "grams",
      "amount": 50
    },
    {
      "name": "garlic",
      "unit": "pieces",
      "amount": 10
    },
    {
      "name": "ginger",
      "unit": "grams",
      "amount": 30
    },
    {
      "name": "pear",
      "unit": "pieces",
      "amount": 1
    },
    {
      "name": "onion",
      "unit": "pieces",
      "amount": 1
    },
    {
      "name": "carrots",
      "unit": "pieces",
      "amount": 2
    },
    {
      "name": "radish",
      "unit": "grams",
      "amount": 300
    },
    {
      "name": "water",
      "unit": "ml",
      "amount": 2000
    },
    {
      "name": "sesame oil",
      "unit": "ml",
      "amount": 20
    },
    {
      "name": "sesame seeds",
      "unit": "teaspoons",
      "amount": 2
    }
  ],
  "instr

## Vslidate JSON output

`jsonschema`

https://python-jsonschema.readthedocs.io/en/stable/

In [12]:
!pip install jsonschema

In [16]:
import json
import jsonschema

In [21]:
# A sample schema, like what we'd get from json.load()
schema_2 = {
    "type" : "object",
    "properties" : {
        "price" : {"type" : "number"},
        "name" : {"type" : "string"},
    },
}

# If no exception is raised by validate(), the instance is valid.
jsonschema.validate(instance={"name" : "Eggs", "price" : 34.99}, schema=schema_2)

# jsonschema.validate(
#   instance={"name" : "Eggs", "price" : "Invalid"}, schema=schema,
# )

In [17]:
receipie_json = json.loads(completion.choices[0].message.function_call.arguments)

In [18]:
receipie_json

{'ingredients': [{'name': 'beef short ribs', 'unit': 'grams', 'amount': 1000},
  {'name': 'soy sauce', 'unit': 'ml', 'amount': 120},
  {'name': 'sugar', 'unit': 'grams', 'amount': 50},
  {'name': 'garlic', 'unit': 'pieces', 'amount': 10},
  {'name': 'ginger', 'unit': 'grams', 'amount': 30},
  {'name': 'pear', 'unit': 'pieces', 'amount': 1},
  {'name': 'onion', 'unit': 'pieces', 'amount': 1},
  {'name': 'carrots', 'unit': 'pieces', 'amount': 2},
  {'name': 'radish', 'unit': 'grams', 'amount': 300},
  {'name': 'water', 'unit': 'ml', 'amount': 2000},
  {'name': 'sesame oil', 'unit': 'ml', 'amount': 20},
  {'name': 'sesame seeds', 'unit': 'teaspoons', 'amount': 2}],
 'instructions': ['Soak the beef short ribs in cold water for about 1 hour to remove the blood.',
  'In a blender, add soy sauce, sugar, garlic, ginger, pear, and onion to make the marinade.',
  'Marinate the beef short ribs with the sauce for about 1 hour.',
  'In a large pot, add the marinated beef short ribs, carrots, radish

In [19]:
jsonschema.validate(receipie_json, schema=schema)

# Convert natural language to DSL

https://www.elastic.co/blog/elasticsearch-prompt-chatgpt-natural-language

In [33]:
es_prompt = 'Given the mapping delimited by triple backticks {} translate the text delimited by triple quotes in a valid Elasticsearch DSL query {}. Give me only the json code part of the answer. Compress the json output removing spaces.'

In [34]:
es_prompt

'Given the mapping delimited by triple backticks {} translate the text delimited by triple quotes in a valid Elasticsearch DSL query {}. Give me only the json code part of the answer. Compress the json output removing spaces.'

In [24]:
es_mapping = {
  "stocks": {
    "mappings": {
      "properties": {
        "close": {"type":"float"},
        "date" : {"type":"date"},
        "high" : {"type":"float"},
        "low"  : {"type":"float"},
        "name" : {
          "type": "text",
          "fields": {
            "keyword":{"type":"keyword", "ignore_above":256}
          }
        },
        "open"  : {"type":"float"},
        "volume": {"type":"long"}
      }
    }
  }
}

In [44]:
query = "Return the first 10 documents of 2017"
user_prompt = es_prompt.format(es_mapping, query)

response = openai.ChatCompletion.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

text = response['choices'][0]['message']['content']

In [45]:
print(text)

{
  "query": {
    "range": {
      "date": {
        "gte": "2017-01-01",
        "lte": "2017-12-31"
      }
    }
  },
  "size": 10
}


In [46]:
query = "Return the first 30 names of all the different stock names"

user_prompt = es_prompt.format(es_mapping, query)

response = openai.ChatCompletion.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

text = response['choices'][0]['message']['content']

print(text)

{
  "size": 0,
  "aggs": {
    "unique_names": {
      "terms": {
        "field": "name.keyword",
        "size": 30
      }
    }
  }
}


In [47]:
query = 'Return the max value of the field "high" for each stock in 2015'

user_prompt = es_prompt.format(es_mapping, query)

response = openai.ChatCompletion.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

text = response['choices'][0]['message']['content']

print(text)

{
  "size": 0,
  "aggs": {
    "stocks": {
      "terms": {
        "field": "name.keyword",
        "size": 10
      },
      "aggs": {
        "max_high": {
          "max": {
            "field": "high"
          }
        }
      }
    }
  },
  "query": {
    "bool": {
      "filter": [
        {
          "range": {
            "date": {
              "gte": "2015-01-01",
              "lte": "2015-12-31"
            }
          }
        }
      ]
    }
  }
}


## Haullacination


In [48]:
query = "Return the first 10 documents of 2017 and 2015"

user_prompt = es_prompt.format(es_mapping, query)

response = openai.ChatCompletion.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

text = response['choices'][0]['message']['content']

print(text)

{
  "query": {
    "bool": {
      "filter": [
        {
          "range": {
            "date": {
              "gte": "2017-01-01",
              "lte": "2017-12-31"
            }
          }
        },
        {
          "range": {
            "date": {
              "gte": "2015-01-01",
              "lte": "2015-12-31"
            }
          }
        }
      ]
    }
  },
  "size": 10
}


In [49]:
query = 'Return the first 10 documents with year 2017 and 2015 in "date" field'

user_prompt = es_prompt.format(es_mapping, query)

response = openai.ChatCompletion.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

text = response['choices'][0]['message']['content']

print(text)

{
  "query": {
    "bool": {
      "filter": [
        {
          "terms": {
            "date": [
              "2017",
              "2015"
            ]
          }
        }
      ]
    }
  },
  "size": 10
}
